In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

df = pd.read_csv("train_en_hi_ru_column_language_12349.csv")
print(f"Dataset size {len(df)} samples")


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device.upper()}")


model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)


model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to(device)

nllb_lang_codes = {'en': 'eng_Latn', 'ru': 'rus_Cyrl', 'hi': 'hin_Deva'}


def translate_batch(texts_list, target_lang_code, batch_size=32):
    translated = []
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang_code)


    for i in tqdm(range(0, len(texts_list), batch_size), desc=f"Translate to {target_lang_code}"):
        batch = texts_list[i:i + batch_size]


        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)


        with torch.no_grad():
            translated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=128
            )


        translated.extend(tokenizer.batch_decode(translated_tokens, skip_special_tokens=True))

    return translated


augmented_data = []


for source_lang in ['en', 'ru', 'hi']:
    source_df = df[df['language'] == source_lang]
    texts_to_translate = source_df['text'].tolist()
    labels = source_df['label'].tolist()


    target_langs = [l for l in ['en', 'ru', 'hi'] if l != source_lang]

    for t_lang in target_langs:
        target_code = nllb_lang_codes[t_lang]


        translated_texts = translate_batch(texts_to_translate, target_code, batch_size=32)


        for text, label in zip(translated_texts, labels):
            augmented_data.append({
                'text': text,
                'label': label,
                'language': t_lang,
                'is_augmented': True
            })


df_aug = pd.DataFrame(augmented_data)
df['is_augmented'] = False
final_df = pd.concat([df, df_aug], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv("train_multi_augmented_37047.csv", index=False)
print(f"\nAugmented dataset: {len(final_df)} rows in 'train_multi_augmented_37047.csv'")

Dataset size 12349 samples
device: CUDA


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translate to rus_Cyrl: 100%|██████████| 110/110 [03:52<00:00,  2.12s/it]



Augmented dataset: 37047 rows in 'train_multi_augmented.csv'
